In [ ]:
import pandas as pd
import matplotlib as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv", index_col=0)
y_train = pd.read_csv("../data/processed/y_train.csv", index_col=0)

X_test = pd.read_csv("../data/processed/X_test.csv", index_col=0)
y_test = pd.read_csv("../data/processed/y_test.csv", index_col=0)

In [ ]:
def clean_data(X):
    X = X.drop(['Patient_ID', 'Diagnosis_Year', 
                'Diagnosis_Date', 'Survival_Months'], axis=1)
    return X

In [ ]:
def engineer_X(X):
    # adding pack_years as a feature (a standard clinical metric)
    X.loc[:, 'Pack_Years'] = (X.loc[:, 'Cigarettes_Per_Day'] / 20) * X.loc[:, 'Years_Smoking']

    # changing cancer stage to ordinal values
    stage_map = {'Stage I': 1,
                 'Stage II': 2,
                 'Stage III': 3,
                 'Stage IV': 4}
    X.loc[:, 'Cancer_Stage_Numeric'] = X.loc[:, 'Cancer_Stage'].map(stage_map)

    # changing binary values to 1 and 0
    binary_cols = ['Secondhand_Smoke', 'Family_History', 'Occupational_Hazard',
                'Chronic_Lung_Disease', 'Asbestos_Exposure', 'Radon_Exposure',
                'Previous_Cancer_History', 'Coughing','Shortness_of_Breath', 
                'Chest_Pain', 'Coughing_Blood', 'Fatigue', 'Weight_Loss', 
                'Wheezing', 'Recurrent_Infections', 'Swallowing_Difficulty', 
                'Finger_Clubbing', 'Metastasis']

    X.loc[:, binary_cols] = X.loc[:, binary_cols].replace({'Yes': 1, 'No':0})

    # converting gender values
    X.loc[:, 'Gender'] = X.loc[:, 'Gender'].replace({'Male': 1, 'Female':0})
    
    # converting cancer types
    X.loc[:, 'Cancer_Type'] = X.loc[:, 'Cancer_Type'].replace({'NSCLC':1, 'SCLC':0})

    # creating a count column for symptoms and risk
    symptom_cols = ['Coughing','Shortness_of_Breath', 'Chest_Pain', 
                    'Coughing_Blood', 'Fatigue', 'Weight_Loss', 'Wheezing', 
                    'Recurrent_Infections', 'Swallowing_Difficulty', 
                    'Finger_Clubbing', 'Metastasis']
    X.loc[:, 'Symptom_Count'] = X.loc[:, symptom_cols].sum(axis=1)

    risk_cols = ['Secondhand_Smoke', 'Family_History', 'Occupational_Hazard',
                'Chronic_Lung_Disease', 'Asbestos_Exposure', 'Radon_Exposure',
                'Previous_Cancer_History']
    X.loc[:, 'Risk_Count'] = X.loc[:, risk_cols].sum(axis=1)

    return X
